# SynBioCrow 2.3 — V2.8 Final General-Backend Recovery
Runs only the two missing general-backend targets (resveratrol, phloretin) using DORAnet, RetroBioCat2, and RetroPath standalone. BioPKS is intentionally excluded because repeated 600-second timeout evidence is already recorded separately.


In [ ]:
# ONE-CELL SELF-CONTAINED RECOVERY
import os,sys,subprocess,shutil,time,json,gzip,urllib.request,importlib,sqlite3,hashlib,zipfile
from pathlib import Path
from google.colab import drive, files

T0=time.time()
def log(msg):
    print(f"[V2.8] elapsed {(time.time()-T0)/60:6.1f} min | {msg}",flush=True)
def run(cmd,*,cwd=None,timeout=3600,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_V2_8")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/manuscript_panel_v2_general_completion")
if REPO.exists(): shutil.rmtree(REPO)
OUT.mkdir(parents=True,exist_ok=True)

log("clone develop/2.3")
run(["git","clone","--depth","1","--branch","develop/2.3","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[doranet,thermo]"],timeout=1200)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
importlib.invalidate_caches()

PANEL=REPO/"benchmarks/2.3/manuscript_panel_v2_general_completion.json"
run([sys.executable,str(REPO/"scripts/validate_benchmark_panel_2_3.py"),str(PANEL),"--expected-target-count","2","--allow-subset"],cwd=REPO)

# RetroBioCat2
log("bootstrap RetroBioCat2")
RBC2_REV="c5f32561b1ed99d0701a6dbb472dc448d6c082e8"
RBC2_ROOT=Path("/content/RetroBioCat-2")
if RBC2_ROOT.exists(): shutil.rmtree(RBC2_ROOT)
run(["git","clone","https://github.com/willfinnigan/RetroBioCat-2.git",str(RBC2_ROOT)],timeout=600)
run(["git","checkout",RBC2_REV],cwd=RBC2_ROOT,timeout=300)
run([sys.executable,"-m","pip","install","-q","PyYAML","sqlalchemy","tables","gdown","tqdm","requests"],timeout=900)
run([sys.executable,"-m","pip","install","-q","--no-deps","--no-build-isolation","-e",str(RBC2_ROOT)],timeout=1200)
os.environ["PYTHONPATH"]=str(RBC2_ROOT)+os.pathsep+os.environ.get("PYTHONPATH","")
if str(RBC2_ROOT) not in sys.path: sys.path.insert(0,str(RBC2_ROOT))
importlib.invalidate_caches()
import rbc2, requests
from rbc2.configs.data_path import path_to_data_folder
db=Path(path_to_data_folder)/"buyability"/"source_mols.db"
db.parent.mkdir(parents=True,exist_ok=True)
def db_ok(path):
    if not path.is_file() or path.stat().st_size<4096: return False
    try:
        con=sqlite3.connect(path)
        ok=con.execute("PRAGMA quick_check").fetchone()[0]=="ok"
        names={r[0] for r in con.execute("select name from sqlite_master where type='table'")}
        con.close()
        return ok and "building_blocks" in names
    except Exception: return False
if not db_ok(db):
    if db.exists(): db.unlink()
    for url in ["https://api.figshare.com/v2/file/download/43860240","https://ndownloader.figshare.com/files/43860240"]:
        try:
            with requests.get(url,stream=True,timeout=180,allow_redirects=True) as resp:
                if resp.status_code not in (200,206): raise RuntimeError(resp.status_code)
                part=db.with_suffix(".part")
                with open(part,"wb") as fh:
                    for chunk in resp.iter_content(1024*1024):
                        if chunk: fh.write(chunk)
                part.replace(db)
            if db_ok(db): break
        except Exception:
            if db.exists(): db.unlink()
assert db_ok(db), "RBC2 database bootstrap failed"

# RetroPath standalone
log("bootstrap bounded RetroPath standalone")
RP=Path("/content/retropath_v2_8"); RP.mkdir(exist_ok=True)
BASE="https://raw.githubusercontent.com/brsynth/retropath2-wrapper/d9948aa1672873d1e3c1905a355f62f714946e67/tests/data"
rules_gz=RP/"rules_d12.csv.gz"; rules=RP/"rules_d12.csv"; sink=RP/"sink.csv"; source=RP/"source.csv"
for dest,url in {rules_gz:BASE+"/rules_d12.csv.gz",sink:BASE+"/lycopene/in/sink.csv",source:BASE+"/lycopene/in/source.csv"}.items():
    urllib.request.urlretrieve(url,dest)
with gzip.open(rules_gz,"rb") as fi, rules.open("wb") as fo: shutil.copyfileobj(fi,fo)
RUNTIME=Path("/content/retropath_runtime_v2_8")
run([sys.executable,str(REPO/"scripts/bootstrap_retropath_standalone.py"),
     "--root",str(RUNTIME),"--rules",str(rules),"--source",str(source),"--sink",str(sink),
     "--smoke-max-steps","2","--smoke-max-structures","20"],timeout=1800)
rpm=json.loads((RUNTIME/"synbiocrow_retropath_standalone.json").read_text())
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EXE"]=rpm["executable"]
os.environ["SYNBIOCROW_RETROPATH_RULES"]=rpm["rules_file"]
os.environ["SYNBIOCROW_RETROPATH_SINK"]=rpm["sink_file"]
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_SMOKE_PASS"]="1"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EQUIVALENCE_PASS"]="1"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_MAX_STEPS"]="2"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_TOPX"]="20"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_TIMEOUT_MINUTES"]="8"

# Reinstall current source, verify readiness
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=600)
from synbiocrow import SynBioCrowEngine
engine=SynBioCrowEngine()
ready=engine.backend_readiness()
print(json.dumps(ready,indent=2,sort_keys=True))
for bid in ("doranet","retrobiocat2","retropath_standalone"):
    assert ready.get(bid,{}).get("available"), f"{bid} unavailable: {ready.get(bid)}"

# Run exactly 2 targets. Output folder is isolated from all prior damaged completion caches.
log("run 2-target general-backend recovery")
cmd=[sys.executable,str(REPO/"scripts/run_benchmark_2_3.py"),
     "--panel",str(PANEL),"--output-dir",str(OUT),
     "--max-route-steps","8","--max-routes","100"]
p=subprocess.Popen(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
for line in p.stdout: print(line.rstrip(),flush=True)
rc=p.wait()
if rc: raise subprocess.CalledProcessError(rc,cmd)

# Package outputs + full state for audit/recovery.
bundle=OUT/"SynBioCrow_2_3_V2_8_GENERAL_RECOVERY_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p!=bundle:
            z.write(p,str(p.relative_to(OUT)))
h=hashlib.sha256(bundle.read_bytes()).hexdigest()
print("BUNDLE",bundle)
print("SHA256",h)
log("COMPLETE")
files.download(str(bundle))
